# Faster R-CNN Detection
Este Notebook es parte de un proyecto que se puede encontrar [aqui](https://github.com/nel-eleven11/Proyecto2_DataScience), donde se busca diseñar una aplicación de datos para comparar e interactuar con diferentes modelos de visión por computadora. Primero, vamos a empezar instalando las librerías requeridas que incluyen 

In [1]:
import polars as pl
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import retinanet_resnet50_fpn
import torchvision.transforms as T
from PIL import Image
import os

## Pre-Procesamiento
A pesar de ya haber realizado un EDA, todavía debemos de preparar los datos en un formato soportado por YoloV8. Primero, vamos a empezar cargando los datos de nuestro dataset.

### Carga de Datos
Al trabajar dentro de Kaggle, podemos importar los datos y los outputs del Notebook de limpieza. Podemos revisar los directorios rápidamente

In [2]:
import os
print(os.listdir("/kaggle/input"))

['mosquito-data', '00-eda-and-cleaning']


Luego, podemos setear algunas variables que nos serán de utilidad para saber dónde se encuentra la información.

In [3]:
RAW_DATA_PATH = "/kaggle/input/mosquito-data"
EDA_OUTPUT_PATH = "/kaggle/input/00-eda-and-cleaning"

print("raw:", os.listdir(RAW_DATA_PATH))
print("eda:", os.listdir(EDA_OUTPUT_PATH))

raw: ['train_images', 'sample_submission_phase1 (1).csv', 'test_images_phase1', 'test_phase1.csv', 'train.csv']
eda: ['__results__.html', 'val.csv', '__notebook__.ipynb', '__results___files', '__output__.json', 'train.csv', 'test.csv', 'custom.css']


Podemos ver por los resultados, que tenemos cargados ya los resultados de la limpieza en EDA_OUTPUT_PATH/train.csv, test.csv y val.csv respectivamente. Adicionalmente, las imágenes que utilizaremos se encuentran en RAW_DATA_PATH/train_images. Podemos cargar los datos hacia DataFrames utilizando Polars.

In [4]:
# Where the training images actually are
image_dir = os.path.join(RAW_DATA_PATH, "train_images")
print("image_dir:", image_dir)

MODEL_OUTPUT_DIR = "/kaggle/working/ConvNext_checkpoints"
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print("MODEL_OUTPUT_DIR:", MODEL_OUTPUT_DIR)

image_dir: /kaggle/input/mosquito-data/train_images
MODEL_OUTPUT_DIR: /kaggle/working/ConvNext_checkpoints


In [5]:
train_df = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "train.csv"))
val_df   = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "val.csv"))
test_df  = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "test.csv"))

print("train shape:", train_df.shape)
print("val shape  :", val_df.shape)
print("test shape :", test_df.shape)

print("Columns:", train_df.columns)
print("Class labels:", train_df.select("class_label").unique())

train shape: (6396, 8)
val shape  : (800, 8)
test shape : (800, 8)
Columns: ['img_fName', 'img_w', 'img_h', 'bbx_xtl', 'bbx_ytl', 'bbx_xbr', 'bbx_ybr', 'class_label']
Class labels: shape: (6, 1)
┌────────────────────┐
│ class_label        │
│ ---                │
│ str                │
╞════════════════════╡
│ aegypti            │
│ japonicus/koreicus │
│ culex              │
│ albopictus         │
│ anopheles          │
│ culiseta           │
└────────────────────┘


Luego del sanity check, podemos confirmar que los datos fueron cargados exitosamente. Ahora, Yolo espera que las clases sean mappeadas de manera numérica.

In [6]:
# ===== Classification mapping (0..N-1, no background) =====
classes = sorted(
    train_df.select("class_label")
            .unique()["class_label"]
            .to_list()
)
print("classes:", classes)

cls_label_to_id = {cls: i for i, cls in enumerate(classes)}  # start at 0
print("cls_label_to_id:", cls_label_to_id)

num_classes_cls = len(classes)
print("num_classes (classification):", num_classes_cls)

train_df = train_df.with_columns(
    pl.col("class_label")
      .replace(cls_label_to_id)
      .cast(pl.Int64)
      .alias("cls_id")
)

val_df = val_df.with_columns(
    pl.col("class_label")
      .replace(cls_label_to_id)
      .cast(pl.Int64)
      .alias("cls_id")
)

test_df = test_df.with_columns(
    pl.col("class_label")
      .replace(cls_label_to_id)
      .cast(pl.Int64)
      .alias("cls_id")
)

classes: ['aegypti', 'albopictus', 'anopheles', 'culex', 'culiseta', 'japonicus/koreicus']
cls_label_to_id: {'aegypti': 0, 'albopictus': 1, 'anopheles': 2, 'culex': 3, 'culiseta': 4, 'japonicus/koreicus': 5}
num_classes (classification): 6


### Carga a Datset de Torch y Transformaciones
El modelo de RetinaNet espera un tipo Dataset de Torch, por lo que debemos transformar nuestros datos ligeramente.

In [7]:
import os
import cv2

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [8]:
CLS_IMG_SIZE = 224

cls_train_tf = A.Compose(
    [
        A.Resize(CLS_IMG_SIZE, CLS_IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

cls_val_tf = A.Compose(
    [
        A.Resize(CLS_IMG_SIZE, CLS_IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

In [9]:
class MosquitoCropClsDataset(Dataset):
    def __init__(self, df_pl, img_dir, transforms=None):
        self.df = df_pl.to_pandas().reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = row["img_fName"]
        img_path = os.path.join(self.img_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        x1 = int(row["bbx_xtl"])
        y1 = int(row["bbx_ytl"])
        x2 = int(row["bbx_xbr"])
        y2 = int(row["bbx_ybr"])

        crop = image[y1:y2, x1:x2, :]

        if self.transforms is not None:
            augmented = self.transforms(image=crop)
            crop = augmented["image"]

        # cls_id is already 0..N-1
        label = int(row["cls_id"])

        return crop, torch.tensor(label, dtype=torch.long)

In [10]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [11]:
train_cls_ds = MosquitoCropClsDataset(
    df_pl=train_df,
    img_dir=image_dir,
    transforms=cls_train_tf,
)

val_cls_ds = MosquitoCropClsDataset(
    df_pl=val_df,
    img_dir=image_dir,
    transforms=cls_val_tf,
)

print(len(train_cls_ds), len(val_cls_ds))

6396 800


In [12]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

train_labels = train_df["cls_id"].to_list()  # 0..N-1

class_counts = Counter(train_labels)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}

sample_weights = [class_weights[cls] for cls in train_labels]
sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

In [13]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_cls_ds,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,  # <-- use sampler, NOT shuffle
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_cls_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [14]:
import torch
import timm  # make sure you did: !pip install timm -q

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# num_classes here should be for CLASSIFICATION: 0..N-1 (no background)
num_classes = len(classes)  # classes = sorted unique class_label

model = timm.create_model(
    "convnext_base",   # or "convnext_tiny" if you want smaller
    pretrained=True,
    num_classes=num_classes,
).to(device)

print("ConvNeXt model ready.")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

device: cuda


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

ConvNeXt model ready.


In [15]:
import torch
import torch.nn as nn

NUM_EPOCHS = 20

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2,
)

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
)

print("Criterion, optimizer and LR scheduler ready.")

Criterion, optimizer and LR scheduler ready.


In [16]:
import time
import copy
import torch
import torch.nn as nn

num_epochs = 20

# Same loss you had
criterion = nn.CrossEntropyLoss()

history = []
start_time = time.time()

num_batches = len(train_loader)
log_interval = max(1, num_batches // 10)

# track best val
best_val_acc = 0.0
best_epoch = 0
best_state_dict = None

print(f"Starting training for {num_epochs} epochs "
      f"({num_batches} batches/epoch, log_interval={log_interval})")

for epoch in range(num_epochs):
    # ===== TRAIN =====
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    epoch_start = time.time()

    print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        running_loss += loss_value

        _, preds = outputs.max(1)
        running_correct += preds.eq(labels).sum().item()
        running_total += labels.size(0)

        # this WILL show in Kaggle logs
        if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == num_batches:
            avg_so_far = running_loss / (batch_idx + 1)
            acc_so_far = running_correct / running_total
            print(
                f"[Epoch {epoch+1}/{num_epochs}] "
                f"Batch {batch_idx+1}/{num_batches} "
                f"- batch_loss: {loss_value:.4f} "
                f"- avg_loss: {avg_so_far:.4f} "
                f"- avg_acc: {acc_so_far:.4f}"
            )

    epoch_time = time.time() - epoch_start
    train_loss = running_loss / num_batches
    train_acc = running_correct / running_total
    current_lr = optimizer.param_groups[0]["lr"]

    # ===== VAL =====
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item()
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / len(val_loader)
    val_acc = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"- train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f} "
        f"| val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f} "
        f"| time: {epoch_time:.1f}s "
        f"| lr: {current_lr:.6f}"
    )

    # ---- track & store best weights ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_state_dict = copy.deepcopy(model.state_dict())
        print(f"  -> New best model at epoch {best_epoch} with val_acc={best_val_acc:.4f}")

    if lr_scheduler is not None:
        lr_scheduler.step()

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "epoch_time_sec": epoch_time,
            "lr": current_lr,
        }
    )

total_time = time.time() - start_time
print(f"\nTotal training time: {total_time/60:.2f} minutes")

# ---- restore best weights ----
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print(f"Restored best model from epoch {best_epoch} with val_acc={best_val_acc:.4f}")
else:
    print("No best_state_dict stored (something went wrong).")

Starting training for 20 epochs (400 batches/epoch, log_interval=40)

===== Epoch 1/20 =====
[Epoch 1/20] Batch 40/400 - batch_loss: 0.7688 - avg_loss: 1.1325 - avg_acc: 0.5766
[Epoch 1/20] Batch 80/400 - batch_loss: 0.3245 - avg_loss: 0.8374 - avg_acc: 0.6969
[Epoch 1/20] Batch 120/400 - batch_loss: 0.3439 - avg_loss: 0.6398 - avg_acc: 0.7703
[Epoch 1/20] Batch 160/400 - batch_loss: 0.3995 - avg_loss: 0.5303 - avg_acc: 0.8098
[Epoch 1/20] Batch 200/400 - batch_loss: 0.3309 - avg_loss: 0.4563 - avg_acc: 0.8369
[Epoch 1/20] Batch 240/400 - batch_loss: 0.1396 - avg_loss: 0.4053 - avg_acc: 0.8560
[Epoch 1/20] Batch 280/400 - batch_loss: 0.1089 - avg_loss: 0.3663 - avg_acc: 0.8692
[Epoch 1/20] Batch 320/400 - batch_loss: 0.1832 - avg_loss: 0.3338 - avg_acc: 0.8818
[Epoch 1/20] Batch 360/400 - batch_loss: 0.1247 - avg_loss: 0.3085 - avg_acc: 0.8910
[Epoch 1/20] Batch 400/400 - batch_loss: 0.0032 - avg_loss: 0.2869 - avg_acc: 0.8990
Epoch [1/20] - train_loss: 0.2869, train_acc: 0.8990 | val_

In [17]:
import torch

torch.save(model.state_dict(), "/kaggle/working/convnext_cls_weights.pth")
print("saved to /kaggle/working/convnext_cls_weights.pth")

saved to /kaggle/working/convnext_cls_weights.pth


In [18]:
import os
import json
import torch

MODEL_OUTPUT_DIR = "/kaggle/working/ConvNext_checkpoints"
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# change 'tiny' to 'base' in the name if you're using convnext_base
run_name = f"convnext_tiny_mosquito_{num_epochs}ep"

# 1) save model weights (state_dict-only)
weights_path = os.path.join(MODEL_OUTPUT_DIR, f"{run_name}_weights.pth")
torch.save(model.state_dict(), weights_path)
print("Saved weights to:", weights_path)

# 2) full checkpoint (weights + optimizer + history)
ckpt_path = os.path.join(MODEL_OUTPUT_DIR, f"{run_name}_full_ckpt.pth")
torch.save(
    {
        "epoch": num_epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "config": {
            "num_classes": num_classes,
            "img_size": CLS_IMG_SIZE,
            "classes": classes,
            "model_name": "convnext_tiny",  # or "convnext_base"
        },
    },
    ckpt_path,
)
print("Saved full checkpoint to:", ckpt_path)

# 3) JSON metadata for later / dashboard stuff
meta_path = os.path.join(MODEL_OUTPUT_DIR, f"{run_name}_meta.json")
with open(meta_path, "w") as f:
    json.dump(
        {
            "run_name": run_name,
            "model_name": "convnext_tiny",   # or "convnext_base"
            "num_epochs": num_epochs,
            "total_train_time_sec": total_time,
            "num_classes": num_classes,
            "img_size": CLS_IMG_SIZE,
            "classes": classes,
            "history": history,
        },

SyntaxError: incomplete input (4026044417.py, line 48)